# 6.6 FA 支持 HiP8 量化方案

## 1. 回顾：HiF8 带来了什么

6.2 节已经建立 HiF8 的格式认知：动态点位域、锥形精度、综合阶码 [−22, 15] 共 38 个（接近 FP16 的 40 个）。站在 FA 量化方案的角度，它最关键的性质是：

> **HiF8 是 8-bit 单数据格式，动态范围接近 FP16，最简只需一个 per-tensor scale，甚至可以免 scale。**

对比 MXFP8：块浮点把动态范围的压力拆给了「每 32 元素一个 E8M0」，代价是 scale 的存储、搬运、分组反量化全套通路；HiF8 用格式本身的动态范围消化了离群值问题，scale 通路**大幅简化**。

## 2. 方案 diff：相对 MXFP8 方案的改动

Kernel 骨架、Tiling 检查、MTE 搬入的改动思路与 6.5 节完全同构（dtype 换成 HiF8、去掉 scale tensor 相关检查），核心差异在 scale 通路：

| 环节 | MXFP8 方案 | HiP8 方案 |
|--|--|--|
| scale 存储 | (S2, D/32) 的 E8M0，数据量的 1/32 | 至多一个 per-tensor scale，常可省 |
| 搬入 | 数据 + scale 双通路 | 单通路 |
| 反量化 | E4M3 × 2^e：指数加/移位 | HiF8 → BF16：解码含动态点位域（硬件解码单元） |
| mm 输入 | 随路反量化成 BF16 / FP8 直吃 | 同左（HiF8 直吃或反量化后 BF16） |
| softmax | FP32 不变 | FP32 不变 |
| 精度管理 | amax 统计频繁（范围窄，18 阶码） | Amax 统计频次可降低（范围接近 FP16） |

工程上意味着：

1. **数据通路更简**：没有分组 scale 的搬入、对齐、分组施加，Tiling 与 Kernel 的量化分支比 MXFP8 干净得多；
2. **精度管理更轻**：E4M3 只有 18 个阶码，动态范围靠细粒度 scaling 撑；HiF8 的 38 个阶码让 delayed-scaling / per-tensor amax 的更新频次显著降低，Vector 上的统计开销随之减少；
3. **训练推理同构**：MX 块浮点训练要按行按列分别量化激活/权重/梯度，HiF8 训练算法更简，训推一致性更好。

## 3. 选型：什么时候用哪个

| 维度 | MXFP8 | HiF8 |
|--|--|--|
| 生态/标准化 | OCP MX 标准，社区广泛 | 昇腾主导，捐献 GCC、IEEE 立项中 |
| 离群值鲁棒性 | gs=32 块 scale 就地关住离群 | 靠锥形精度 + 宽动态范围 |
| scale 通路 | 有（1/32 数据量 + 分组反量化） | 近乎无 |
| 硬件支持 | 已上市硬件支持 | 新一代昇腾硬件支持 |
| 适用建议 | 离群分布极端、需要与 MX 生态对齐 | 追求最简数据流、宽动态范围 |

> 具体某代硬件支持哪种格式、指令细节如何，以对应 CANN 版本官方文档与算子库（FusedInferAttentionScore 的量化分支）为准——**课程要传授的是「格式决定数据通路」的分析方法**：拿到一种新格式（未来的 HiF4、mxFP6 同理），按「位布局 → 动态范围 → scale 需求 → 搬入/反量化/mm 三处 diff」四步推演，就能推出它的 FA 方案形态。

## 小测验

1. HiF8 相对 MXFP8 在 scale 通路上的核心优势是什么？代价是什么？
2. E4M3 只有 18 个阶码、HiF8 有 38 个（FP16 是 40 个）。这如何转化为量化方案里 amax 统计频次的开销差异？
3. 拿到一种全新的 8-bit 格式，按本章方法推演它的 FA 量化方案要走哪四步？
4. 结合第 5 章与本章：非量化 FA → MXFP8 FA → HiF8 FA，哪一层骨架始终没变？量化每次改的都是什么？